In [0]:
dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp")
dbutils.widgets.text("table_name", "", "Table Name")
dbutils.widgets.text("load_type", "")
dbutils.widgets.text("merge_keys", "")

landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
load_type=dbutils.widgets.get("load_type")
merge_keys=dbutils.widgets.get("merge_keys")

In [0]:
source_table = f"azuredb_caresync.bronze.{table_name}"
filter_condition = f"landing_timestamp = '{landing_timestamp}'"

print("=" * 80)
print("STEP 2: Reading bronze data")
print("=" * 80)
print(f"  Source table : {source_table}")
print(f"  Filter       : {filter_condition}")
print("  Equivalent query:")
print(f"    SELECT * FROM {source_table} WHERE {filter_condition}")

bronze_df = spark.read.table(source_table).filter(filter_condition)

print(f"  Rows read from bronze : {bronze_df.count()}")
print(f"  Columns ({len(bronze_df.columns)}) : {bronze_df.columns}")

In [0]:
from pyspark.sql import functions as F

print("=" * 80)
print("STEP 3: Adding audit columns")
print("=" * 80)

updated_df=bronze_df.withColumn("insert_timestamp" ,F.current_timestamp()).withColumn("updated_timestamp", F.current_timestamp())

print("  Added columns : insert_timestamp, updated_timestamp (current_timestamp())")
print(f"  Final columns ({len(updated_df.columns)}) : {updated_df.columns}")

In [0]:
from delta.tables import DeltaTable

target_table = f"azuredb_caresync.silver.{table_name}"
load_type_upper = load_type.strip().upper()

print("=" * 80)
print("STEP 4: Writing to silver")
print("=" * 80)
print(f"  Target table : {target_table}")
print(f"  Load type    : {load_type_upper}")

if load_type_upper == "FULL":
    print("  Mode: OVERWRITE - existing silver data will be replaced")
    print("  Equivalent query:")
    print(f"    CREATE OR REPLACE TABLE {target_table} AS SELECT * FROM <source_batch>")
    updated_df.write.mode("overwrite").saveAsTable(target_table)
    print("  FULL load completed.")

elif load_type_upper == "APPEND":
    print("  Mode: APPEND - rows will be added to existing silver data")
    print("  Equivalent query:")
    print(f"    INSERT INTO {target_table} SELECT * FROM <source_batch>")
    updated_df.write.mode("append").saveAsTable(target_table)
    print("  APPEND load completed.")

elif load_type_upper == "MERGE":
    keys = [k.strip() for k in merge_keys.split(",") if k.strip()]
    print(f"  Merge keys parsed : {keys}")
    if not keys:
        raise ValueError("merge_keys must be provided (comma-separated) when load_type is MERGE")

    if not spark.catalog.tableExists(target_table):
        # First load: create the target table
        print(f"  Target table does not exist - creating it with the current batch (initial load)")
        print("  Equivalent query:")
        print(f"    CREATE TABLE {target_table} AS SELECT * FROM <source_batch>")
        updated_df.write.mode("overwrite").saveAsTable(target_table)
        print("  Initial load completed.")
    else:
        print("  Target table exists - performing MERGE (upsert)")
        merge_condition = " AND ".join([f"t.`{k}` = s.`{k}`" for k in keys])
        # On update, keep the original insert_timestamp of the existing record
        update_set = {f"`{c}`": f"s.`{c}`" for c in updated_df.columns if c != "insert_timestamp"}
        insert_set = {f"`{c}`": f"s.`{c}`" for c in updated_df.columns}

        print(f"  Merge condition : {merge_condition}")
        print("  Columns updated on match (insert_timestamp preserved):")
        print(f"    {list(update_set.keys())}")

        update_sql = ",\n      ".join([f"t.{k} = {v}" for k, v in update_set.items()])
        insert_cols = ", ".join(insert_set.keys())
        insert_vals = ", ".join(insert_set.values())
        merge_sql = (
            f"MERGE INTO {target_table} AS t\n"
            f"USING <source_batch> AS s\n"
            f"ON {merge_condition}\n"
            f"WHEN MATCHED THEN UPDATE SET\n      {update_sql}\n"
            f"WHEN NOT MATCHED THEN INSERT ({insert_cols})\n"
            f"  VALUES ({insert_vals})"
        )
        print("  Equivalent MERGE query:")
        print("    " + merge_sql.replace("\n", "\n    "))

        target_dt = DeltaTable.forName(spark, target_table)
        (target_dt.alias("t")
            .merge(updated_df.alias("s"), merge_condition)
            .whenMatchedUpdate(set=update_set)
            .whenNotMatchedInsert(values=insert_set)
            .execute())

        # Report merge metrics from the Delta history of the last operation
        metrics = target_dt.history(1).select("operationMetrics").collect()[0][0] or {}
        print("  MERGE completed. Metrics:")
        print(f"    Source rows     : {metrics.get('numSourceRows')}")
        print(f"    Rows inserted   : {metrics.get('numTargetRowsInserted')}")
        print(f"    Rows updated    : {metrics.get('numTargetRowsUpdated')}")
        print(f"    Rows unchanged  : {metrics.get('numTargetRowsCopied')}")

else:
    raise ValueError(f"Unsupported load_type '{load_type}'. Expected FULL, APPEND or MERGE.")

In [0]:
record_count = updated_df.count()

print("=" * 80)
print("STEP 5: Finished")
print("=" * 80)
print(f"  Records processed : {record_count}")
print(f"  Silver row count  : {spark.read.table(f'azuredb_caresync.silver.{table_name}').count()}")
dbutils.notebook.exit(str(record_count))